# Deep Learning–Based Classification of Imbalanced WCE Datasets

This notebook follows **all tasks from MINIPROJECT.pdf** for classifying gastrointestinal diseases on imbalanced Wireless Capsule Endoscopy (WCE) data using PyTorch.

> Recommended runtime: **Google Colab with GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
# If needed in Colab, uncomment and run:
# !pip -q install torch torchvision scikit-learn seaborn pandas matplotlib tqdm

In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Global Configuration
Update `DATASET_ROOT` to your dataset folder. The expected structure is:

`DATASET_ROOT/class_name/image1.jpg`

In [ ]:
DATASET_ROOT = Path("/content/kvasir-capsule")   # change this path in Colab
OUTPUT_DIR = Path("/content/wce_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 5          # increase (e.g., 10-25) for stronger performance
UNDERSAMPLE_THRESHOLD = 200
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4  # L2 regularization

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## Utility Functions

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def build_metadata(root: Path) -> pd.DataFrame:
    rows = []
    if not root.exists():
        raise FileNotFoundError(f"Dataset path does not exist: {root}")

    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    for class_dir in class_dirs:
        for fp in class_dir.rglob("*"):
            if fp.is_file() and fp.suffix.lower() in IMG_EXTS:
                rows.append({"filepath": str(fp), "label": class_dir.name})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No images found. Check folder structure and file extensions.")
    return df


def plot_distribution(df: pd.DataFrame, title: str, save_name: str = None):
    counts = df["label"].value_counts().sort_values(ascending=False)
    ax = counts.plot(kind="bar", color="steelblue")
    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of images")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    if save_name:
        plt.savefig(OUTPUT_DIR / save_name, dpi=200)
    plt.show()


def stratified_split(df: pd.DataFrame, seed: int = 42):
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df["label"],
        random_state=seed,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=seed,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def random_under_sample(df: pd.DataFrame, threshold: int, seed: int = 42):
    sampled_parts = []
    rng = np.random.default_rng(seed)
    for _, part in df.groupby("label"):
        if len(part) > threshold:
            idx = rng.choice(part.index.values, size=threshold, replace=False)
            sampled_parts.append(part.loc[idx])
        else:
            sampled_parts.append(part)
    out = pd.concat(sampled_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out


def build_augmented_to_threshold(df: pd.DataFrame, threshold: int, seed: int = 42):
    # Returns dataframe with additional boolean column `augmented`
    base = df.copy()
    base["augmented"] = False
    rng = np.random.default_rng(seed)

    extras = []
    counts = base["label"].value_counts().to_dict()
    for label, count in counts.items():
        if count < threshold:
            need = threshold - count
            label_rows = base[base["label"] == label]
            sampled_idx = rng.choice(label_rows.index.values, size=need, replace=True)
            extra = label_rows.loc[sampled_idx].copy()
            extra["augmented"] = True
            extras.append(extra)

    if extras:
        out = pd.concat([base] + extras).sample(frac=1, random_state=seed).reset_index(drop=True)
    else:
        out = base.sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

## Task 1: Dataset Exploration & Imbalance Analysis
1. Load WCE dataset
2. Plot class distribution
3. Identify majority/minority classes
4. Add explanation (5–6 lines)

In [ ]:
metadata_df = build_metadata(DATASET_ROOT)
print(f"Total images: {len(metadata_df)}")
print(f"Total classes: {metadata_df['label'].nunique()}")
display(metadata_df.head())

plot_distribution(metadata_df, "Original Class Distribution", "task1_original_distribution.png")

class_counts = metadata_df["label"].value_counts().sort_values(ascending=False)
majority_class = class_counts.index[0]
minority_class = class_counts.index[-1]

print("Majority class:", majority_class, "(", class_counts.iloc[0], "images )")
print("Minority class:", minority_class, "(", class_counts.iloc[-1], "images )")
print("
Class counts:
", class_counts)

### Task 1 Explanation (5–6 lines)
The class-distribution plot shows a clear imbalance across gastrointestinal disease categories.
A few majority classes dominate the dataset, while minority classes have substantially fewer samples.
In medical diagnosis tasks, this imbalance can bias a model toward frequent classes and reduce sensitivity on rare findings.
Missing rare but clinically critical patterns can lead to unsafe predictions in real-world screening workflows.
Therefore, balancing strategies are required to improve fairness across classes and increase macro-level metrics such as recall and F1.

## Task 2: Under-Sampling
- Randomly reduce majority classes to a fixed threshold (`UNDERSAMPLE_THRESHOLD`)
- Keep all minority samples
- Plot updated distribution and add observations

In [ ]:
under_df_all = random_under_sample(metadata_df, threshold=UNDERSAMPLE_THRESHOLD, seed=SEED)
print("Original dataset size:", len(metadata_df))
print("Under-sampled dataset size:", len(under_df_all))

plot_distribution(under_df_all, f"Under-Sampled Distribution (threshold={UNDERSAMPLE_THRESHOLD})", "task2_undersampled_distribution.png")

### Task 2 Observation
Under-sampling effectively controls dominance of very frequent classes and makes class counts more balanced.
The trade-off is that some majority-class images are discarded, which can remove useful visual variability.
This improves balance-driven learning but may reduce representation of benign intra-class diversity.
Hence, under-sampling is often combined with augmentation to recover diversity while preserving class fairness.

## Task 3: Data Augmentation–Based Over-Sampling (Minority Classes Only)
Required augmentations:
- Horizontal flip
- Rotation (±20°)
- Width/Height shift (0.2)
- Zoom (0.2)

We first under-sample, then over-sample minority classes to the same threshold using augmentation.

In [ ]:
minority_aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2)),
])

under_aug_df_all = build_augmented_to_threshold(under_df_all, threshold=UNDERSAMPLE_THRESHOLD, seed=SEED)
print("Under-sampled size:", len(under_df_all))
print("Under-sampled + augmentation-expanded size:", len(under_aug_df_all))

plot_distribution(under_aug_df_all, f"Under-sampled + Augmented Distribution (threshold={UNDERSAMPLE_THRESHOLD})", "task3_under_aug_distribution.png")

print("
Dataset size summary")
print("Original:", len(metadata_df))
print("Under-sampled:", len(under_df_all))
print("Under-sampled + augmentation-expanded:", len(under_aug_df_all))

In [ ]:
counts_under = under_df_all["label"].value_counts()
minority_labels = counts_under[counts_under < UNDERSAMPLE_THRESHOLD].index.tolist()
if len(minority_labels) == 0:
    minority_labels = [counts_under.index[-1]]

sample_label = minority_labels[0]
sample_path = under_df_all[under_df_all["label"] == sample_label].iloc[0]["filepath"]
img = Image.open(sample_path).convert("RGB")

augmented_images = [minority_aug_transform(img) for _ in range(4)]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(img.resize((IMG_SIZE, IMG_SIZE)))
axes[0].set_title("Original")
axes[0].axis("off")

for i, aug_img in enumerate(augmented_images, start=1):
    axes[i].imshow(aug_img)
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")

plt.suptitle(f"Before/After Augmentation Samples ({sample_label})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "task3_before_after_augmentation.png", dpi=200)
plt.show()

## Task 4: Data Preprocessing
- Resize to **224×224**
- Normalize pixel values to **[0,1]**
- Split into **Train 70% / Validation 15% / Test 15%**

> Best-practice note: imbalance handling is applied on **training data only** for model training, while validation/test remain untouched for fair evaluation.

In [ ]:
train_df_raw, val_df, test_df = stratified_split(metadata_df, seed=SEED)

print("Split sizes:")
print("Train:", len(train_df_raw), f"({len(train_df_raw)/len(metadata_df):.2%})")
print("Validation:", len(val_df), f"({len(val_df)/len(metadata_df):.2%})")
print("Test:", len(test_df), f"({len(test_df)/len(metadata_df):.2%})")

class_names = sorted(metadata_df["label"].unique().tolist())
label_to_idx = {c: i for i, c in enumerate(class_names)}
print("
Classes:", class_names)

In [ ]:
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2)),
    transforms.ToTensor(),
])

class ImageTableDataset(Dataset):
    def __init__(self, df: pd.DataFrame, label_to_idx: dict, base_transform, aug_transform=None):
        self.df = df.reset_index(drop=True).copy()
        if "augmented" not in self.df.columns:
            self.df["augmented"] = False
        self.label_to_idx = label_to_idx
        self.base_transform = base_transform
        self.aug_transform = aug_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        use_aug = bool(row.get("augmented", False)) and (self.aug_transform is not None)
        x = self.aug_transform(img) if use_aug else self.base_transform(img)
        y = self.label_to_idx[row["label"]]
        return x, y


def make_loader(df, is_train=False, shuffle=False):
    ds = ImageTableDataset(
        df=df,
        label_to_idx=label_to_idx,
        base_transform=base_transform,
        aug_transform=train_aug_transform if is_train else None,
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

## Task 5: Transfer Learning with Three Models
Models used:
1. **EfficientNet-B0**
2. **MobileNetV3-Small**
3. **ResNet101**

For each model:
- load ImageNet weights
- freeze early layers
- replace classifier head
- add Dropout and L2 regularization

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen


def freeze_early_layers_sequential(features_module, freeze_ratio=0.7):
    layers = list(features_module.children())
    n_freeze = int(len(layers) * freeze_ratio)
    for layer in layers[:n_freeze]:
        for p in layer.parameters():
            p.requires_grad = False


def build_model(model_name: str, num_classes: int, dropout_p=0.4):
    if model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        freeze_early_layers_sequential(model.features, freeze_ratio=0.7)
        in_features = model.classifier[-1].in_features
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_p), nn.Linear(in_features, num_classes))

    elif model_name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        freeze_early_layers_sequential(model.features, freeze_ratio=0.7)
        in_features = model.classifier[-1].in_features
        model.classifier = nn.Sequential(
            nn.Linear(model.classifier[0].in_features, 1024),
            nn.Hardswish(),
            nn.Dropout(p=dropout_p),
            nn.Linear(1024, num_classes),
        )

    elif model_name == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        for p in model.conv1.parameters():
            p.requires_grad = False
        for p in model.bn1.parameters():
            p.requires_grad = False
        for p in model.layer1.parameters():
            p.requires_grad = False
        for p in model.layer2.parameters():
            p.requires_grad = False

        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p), nn.Linear(in_features, num_classes))
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model


model_names = ["efficientnet_b0", "mobilenet_v3_small", "resnet101"]
for mn in model_names:
    m = build_model(mn, num_classes=len(class_names))
    total, trainable, frozen = count_params(m)
    print(f"
Model: {mn}")
    print(m)
    print(f"Total params: {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Frozen params: {frozen:,}")

## Task 6: Intelligent Learning Rate Control
We use **ReduceLROnPlateau** based on validation loss and track learning-rate value each epoch.

In [ ]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss = 0.0
    all_preds, all_targets = [], []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(yb.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc


def fit_model(model_name, train_loader, val_loader, epochs=EPOCHS):
    model = build_model(model_name, num_classes=len(class_names)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_wts = copy.deepcopy(model.state_dict())
    best_val = float("inf")

    for epoch in range(epochs):
        train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, val_acc = run_one_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        if val_loss < best_val:
            best_val = val_loss
            best_wts = copy.deepcopy(model.state_dict())

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, train_acc={train_acc:.4f}, val_acc={val_acc:.4f}, lr={current_lr:.6f}")

    model.load_state_dict(best_wts)
    return model, history


def evaluate_model(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(yb.numpy())

    acc = accuracy_score(all_targets, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)

    return {"accuracy": acc, "precision_macro": prec, "recall_macro": rec, "f1_macro": f1, "cm": cm}


def plot_lr_and_loss(history, prefix):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(epochs, history["lr"], marker="o")
    axes[0].set_title("Learning Rate vs Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Learning Rate")

    axes[1].plot(epochs, history["train_loss"], marker="o", label="Train Loss")
    axes[1].plot(epochs, history["val_loss"], marker="o", label="Validation Loss")
    axes[1].set_title("Training vs Validation Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{prefix}_lr_loss_curves.png", dpi=200)
    plt.show()


def plot_confusion(cm, title, save_name, class_names):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / save_name, dpi=200)
    plt.show()

## Task 7: Training & Evaluation Under Three Imbalance Settings
Settings:
1. **No imbalance handling**
2. **Under-sampling only**
3. **Under-sampling + augmentation**

For fairness, validation and test sets are unchanged across settings.

In [ ]:
train_raw = train_df_raw.copy()
train_raw["augmented"] = False

train_under = random_under_sample(train_df_raw, threshold=UNDERSAMPLE_THRESHOLD, seed=SEED)
train_under["augmented"] = False

train_under_aug = build_augmented_to_threshold(train_under, threshold=UNDERSAMPLE_THRESHOLD, seed=SEED)

print("Training size by setting:")
print("Raw:", len(train_raw))
print("Under-sampling:", len(train_under))
print("Under-sampling + augmentation:", len(train_under_aug))

In [ ]:
val_loader = make_loader(val_df.assign(augmented=False), is_train=False, shuffle=False)
test_loader = make_loader(test_df.assign(augmented=False), is_train=False, shuffle=False)

train_tables = {
    "no_handling": train_raw,
    "under_sampling": train_under,
    "under_plus_aug": train_under_aug,
}

all_results = []
all_histories = {}

for setting_name, train_table in train_tables.items():
    print(f"
===== Setting: {setting_name} =====")
    train_loader = make_loader(train_table, is_train=True, shuffle=True)

    for model_name in model_names:
        print(f"
--- Training {model_name} on {setting_name} ---")
        model, history = fit_model(model_name, train_loader, val_loader, epochs=EPOCHS)
        metrics = evaluate_model(model, test_loader)

        all_histories[(setting_name, model_name)] = history

        all_results.append({
            "setting": setting_name,
            "model": model_name,
            "accuracy": metrics["accuracy"],
            "precision_macro": metrics["precision_macro"],
            "recall_macro": metrics["recall_macro"],
            "f1_macro": metrics["f1_macro"],
            "cm": metrics["cm"],
        })

        plot_lr_and_loss(history, prefix=f"{setting_name}_{model_name}")

results_df = pd.DataFrame(all_results).drop(columns=["cm"])
results_df = results_df.sort_values(["setting", "f1_macro"], ascending=[True, False]).reset_index(drop=True)
print("
Comparison Table")
display(results_df)
results_df.to_csv(OUTPUT_DIR / "task7_comparison_table.csv", index=False)

In [ ]:
best_rows = (
    pd.DataFrame(all_results)
    .sort_values(["setting", "f1_macro"], ascending=[True, False])
    .groupby("setting", as_index=False)
    .first()
)

display(best_rows[["setting", "model", "accuracy", "precision_macro", "recall_macro", "f1_macro"]])

for _, row in best_rows.iterrows():
    setting = row["setting"]
    model_name = row["model"]
    cm = row["cm"]
    plot_confusion(cm, title=f"Confusion Matrix | {setting} | {model_name}", save_name=f"task7_cm_{setting}_{model_name}.png", class_names=class_names)

### Task 7 Analysis (8–10 lines)
Across the three training settings, class-imbalance handling typically improves macro-level metrics.
The no-handling baseline often achieves acceptable overall accuracy but underperforms on minority classes.
Under-sampling reduces majority-class bias and usually increases macro recall, though some information is discarded.
Combining under-sampling with augmentation generally offers the best trade-off by restoring data diversity.
The comparison table and confusion matrices should be interpreted together: improved minority-class recall is the key target.
Learning-rate adaptation stabilizes optimization, especially when validation loss plateaus.
Transfer learning with frozen early layers reduces overfitting risk and lowers training time compared with full fine-tuning.
Among the three backbones, deeper models may reach higher ceiling performance while lightweight models train faster.
Final model choice should balance F1-macro, inference cost, and clinical safety priorities.

## Comparative Model Summary (Task 5 Deliverable)

In [ ]:
summary_rows = []
for mn in model_names:
    model = build_model(mn, num_classes=len(class_names))
    total, trainable, frozen = count_params(model)
    summary_rows.append({
        "model": mn,
        "total_params": total,
        "trainable_params": trainable,
        "frozen_params": frozen,
        "trainable_ratio": trainable / total,
    })

model_summary_df = pd.DataFrame(summary_rows)
display(model_summary_df)
model_summary_df.to_csv(OUTPUT_DIR / "task5_model_trainable_vs_frozen.csv", index=False)

## Final Notes
- All required tasks (1–7) are implemented in this notebook.
- Generated figures and tables are saved in `OUTPUT_DIR` for report inclusion.
- Increase `EPOCHS` and tune batch size / learning rate for stronger final performance.